In [ ]:
!pip install -U peft bitsandbytes transformers accelerate
!pip install -U trl
!pip install PyMuPDF
!pip install -U torchao peft

In [ ]:
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig, SFTTrainer
import torch, gc
import zipfile
import fitz
import re

dataset = load_dataset("roneneldan/TinyStories", split="train")
dataset, dataset[0]

In [ ]:
def extract_text_from_pdf(pdf_path):
  text_blocks = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_blocks.append(text)
  return "\n".join(text_blocks)

pdf_texts = extract_text_from_pdf("/content/Metformin.pdf")
print(pdf_texts)

In [ ]:
def split_paragraphs(pages):
  paragraphs = []
  for page_text in pages:
    chunks = re.split(r'\n\s*\n', page_text)
    for chunk in chunks:
      clean = chunk.strip()
      if len(clean) > 30:
        paragraphs.append(clean)
  return paragraphs

paragraphs = split_paragraphs(pdf_texts)
data = [{"text": p} for p in paragraphs]
data

In [ ]:
import re
import fitz
from datasets import Dataset

dataset = Dataset.from_list(data)
print(f"Dataset created with {len(dataset)} rows.")
dataset

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
  tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [ ]:
if not data:
    paragraphs = [p.strip() for p in pdf_texts.split('\n\n') if len(p.strip()) > 30]
    data = [{"text": p} for p in paragraphs]
    from datasets import Dataset
    dataset = Dataset.from_list(data)

if len(dataset) > 0:
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    model = AutoModelForCausalLM.from_pretrained(model_name)
    print(tokenized)
else:
    print("Error: Dataset is empty. Please check the PDF extraction step.")

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [ ]:
help(TrainingArguments)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)
trainer.train()

### **Feerze some layer and finetune unfreeze layer(old CNN and Bert sytel method)**

In [ ]:
for param in model.parameters():
  param.requires_grad = False

for name, param in model.named_parameters():
  if any(f"layers.{i}." in name for i in range(20, 24)):
    param.requires_grad = True
  if "lm_head" in name:
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/total*100:.2f}% of total parameters")

In [ ]:
from transformers import DataCollatorForLanguageModeling

dataset = load_dataset("json", data_files={"train": "pharma_non_instruction.jsonl"})
data_collector = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir="./tinyllama-pharma-last4",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_steps=200,
    save_total_limit=2,
    logging_steps=20,
    learning_rate=1e-4,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collector
)
trainer.train()
trainer.save_model("./tinyllama-pharma-last4-final")

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
non_inst_model_lora = get_peft_model(model, lora_config)
non_inst_model_lora.to("cuda")

args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=1,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)
trainer.train()

In [ ]:
import zipfile
import os

zip_path = "/content/tinyllama-non-instruction.zip"
extract_path = "/content/tinyllama-non-instruction"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Extracted to: {extract_path}")
print(os.listdir(extract_path))

In [ ]:
import os
from transformers import AutoModelForCausalLM

model_path = "/content/tinyllama-non-instruction/checkpoint-5"

if os.path.exists(model_path):
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=quantization_config,
        device_map="auto"
    )
    print("Model loaded successfully from checkpoint folder.")
else:
    print(f"Error: {model_path} not found. Please check the extraction output.")

prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
dataset = load_dataset("HuggingFaceFW/fineweb")
pubmed = load_dataset("ncbi/pubmed")
dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer")
dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
owt = load_dataset("Skylion007/openwebtext")
ds = load_dataset("armanc/scientific_papers")

In [ ]:
https://huggingface.co/datasets/Amod/mental_health_counseling_conversations
https://huggingface.co/datasets/yahma/alpaca-cleaned
https://huggingface.co/datasets/Open-Orca/OpenOrca
https://huggingface.co/datasets/tatsu-lab/alpaca
https://huggingface.co/datasets/OpenAssistant/oasst1

In [ ]:
https://huggingface.co/datasets/Anthropic/hh-rlhf
https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned
https://huggingface.co/datasets/xinlai/Math-Step-DPO-10K

### **Instruction Finetuning on Domain Specific**

In [ ]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
zip_path = "/content/tinyllama-lora.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
model_path = "/content/tinyllama-non-instruction/checkpoint-5"
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
dataset

In [ ]:
def format_row(example):
  question = example["Context"]
  answer = example["Response"]
  example["Text"] = f"[Context] {question} [/Response] {answer}"
  return example

formatted_dataset = dataset.map(format_row)
formatted_dataset, formatted_dataset[0]["Text"]

In [ ]:
import pandas as pd

df = pd.DataFrame(formatted_dataset)
df

In [ ]:
df.to_csv("mental_health_counseling_conversations.csv", index=False)
df.to_json("mental_health_counseling_conversations.jsonl", orient="records", lines=True)

In [ ]:
dataset = load_dataset("csv", data_files="mental_health_counseling_conversations.csv",split="train")
dataset

In [ ]:
dataset = load_dataset("json", data_files="mental_health_counseling_conversations.jsonl", split="train")
dataset

In [ ]:
def format_example(example):
  prompt = f"### Instruction:\nAnalyze the following mental health query.\n### Input:\n{example['Context']}\n### Response:\n{example['Response']}"
  return {"text": prompt}

dataset = dataset.map(format_example)
tokenized = dataset.map(tokenize_fn, batched=True)
dataset, dataset['text'][0]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
instruction_model_lora = get_peft_model(non_instruction_model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora-instruction",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=1,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=instruction_model_lora,
    args=args,
    train_dataset=tokenized
)
trainer.train()

In [ ]:
model_path = "/content/tinyllama-instruction/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
prompt = "Explain the mechanism of action of Metformin."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")

def format_row(example):
  prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
  return {"text": prompt}

dataset = dataset.map(format_row)
tokenized = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_and_mask(example):
  text=example["text"]
  enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
  input_ids = enc["input_ids"]

  response_marker = "### Response:"
  response_start = text.find(response_marker)

In [ ]:
questions = [
    "Explain the mechanism of action of Metformin.",
    "List two advantages of combining Atorvastatin with Ezetimibe.",
    "Summarize how mRNA vaccines work and mention one current research focus."
]

In [ ]:
for q in questions:
    print("Question:", q)
    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = non_instruction_model.generate(**inputs, max_new_tokens=80)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = instruction_model.generate(**inputs, max_new_tokens=100)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("="*80, "\n")

### **Preference Optimization Technique: RLHF, RL, AIF, DPO(Direct preference Optimization)**



> base_model--> non_instruction_model--> instruction_model--> preference_model



In [ ]:
zip_path = "/content/tinyllama-instruction.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

instruction_checkpoint = "/content/checkpoint-3"
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = AutoModelForCausalLM.from_pretrained(base_model, device_map="auto")
model = PeftModel.from_pretrained(model, instruction_checkpoint)
model = model.merge_and_unload()

preference_model = get_peft_model(model, lora_config)

In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    beta=0.1,
    report_to="none",
    loss_type="sigmoid",
    remove_unused_columns=False
)

In [ ]:
trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

In [ ]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

zip_file = "/content/tinyllama-non-instruction.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
inputs = tokenizer(question, return_tensors="pt").to("cuda")
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
zip_path = "/content/tinyllama-preference-alignment.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

inputs = tokenizer(question, return_tensors="pt").to("cuda")
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)
preference_aligned_model.to("cuda")

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))